In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Titanic-Dataset.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
print("Shape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:\n", df.dtypes)

Shape: (891, 12)

Missing values per column:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Duplicate rows: 0

Data types:
 PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


## Data Quality Report

The dataset has 891 rows and 12 columns with no duplicate rows. Three columns have missing values: Age (177 missing, ~20%), Cabin (687 missing, ~77%), and Embarked (2 missing, <1%). Given Cabin's extremely high missing rate, it will likely need to be dropped or transformed rather than imputed. Age's moderate gap can be reasonably filled. Embarked's 2 missing values can simply use the mode.

In [3]:
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)
print(f"Filled {177} missing Age values with median: {median_age}")

Filled 177 missing Age values with median: 28.0


Age is filled with the median (rather than mean) because age distributions often have outliers (very young infants, elderly passengers) that would skew the mean. The median is more robust to this and preserves a realistic typical age.

In [4]:
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)
print(f"Filled 2 missing Embarked values with mode: {mode_embarked}")

Filled 2 missing Embarked values with mode: S


Embarked has only 2 missing values out of 891 — too few to warrant a complex strategy. Filling with the mode (most common boarding port) is standard practice for such a small gap and won't meaningfully affect the data.

In [5]:
df['Has_Cabin'] = df['Cabin'].notnull().astype(int)
df = df.drop('Cabin', axis=1)
print("Created 'Has_Cabin' feature and dropped original Cabin column")

Created 'Has_Cabin' feature and dropped original Cabin column


Cabin is missing for 77% of passengers — too much to reliably impute, and guessing cabin numbers would introduce fabricated data. Instead of dropping the column entirely, we extract the information that IS meaningful: whether or not a cabin was recorded. This becomes a new binary feature 'Has_Cabin', which may correlate with passenger class or survival, before dropping the original text column.

In [6]:
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicate rows found and removed: {duplicates_before}")

Duplicate rows found and removed: 0


No duplicate rows were found in this dataset (0 out of 891). The code above is included to demonstrate the standard duplicate-check process, even though this dataset happened to be clean on this front.

In [7]:
print("Unique Sex values:", df['Sex'].unique())
print("Unique Embarked values:", df['Embarked'].unique())
print("Unique Pclass values:", df['Pclass'].unique())

Unique Sex values: <StringArray>
['male', 'female']
Length: 2, dtype: str
Unique Embarked values: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str
Unique Pclass values: [3 1 2]


Categorical columns (Sex, Embarked, Pclass) already show consistent formatting — no mixed-case or abbreviation issues found, so no standardisation was needed here. Order Date-style inconsistencies don't apply to this dataset.

In [8]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Fare'] < lower_bound) | (df['Fare'] > upper_bound)]
print(f"Fare outliers found: {len(outliers)}")
print(f"Normal range: {lower_bound:.2f} to {upper_bound:.2f}")
print(f"Max fare in data: {df['Fare'].max()}")

Fare outliers found: 116
Normal range: -26.72 to 65.63
Max fare in data: 512.3292


116 Fare outliers were detected using the IQR method (values above 65.63). Rather than capping or removing these, we retain them because they represent legitimate first-class fares rather than data errors — high fares are expected given the range of ticket classes on the Titanic. Removing them would discard real information relevant to survival analysis (fare likely correlates with class and cabin location).

In [9]:
print(df.dtypes)

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
Has_Cabin        int64
dtype: object


In [10]:
summary = pd.DataFrame({
    'Metric': ['Rows', 'Null count (Age)', 'Null count (Cabin)', 'Null count (Embarked)', 'Duplicate rows', 'Columns'],
    'Before': [891, 177, 687, 2, 0, 12],
    'After': [len(df), df['Age'].isnull().sum(), 'N/A (replaced with Has_Cabin)', df['Embarked'].isnull().sum(), df.duplicated().sum(), df.shape[1]]
})
summary

,Metric,Before,After
0,Rows,891,891
1,Null count (Age),177,0
2,Null count (Cabin),687,N/A (replaced with Has_Cabin)
3,Null count (Embarked),2,0
4,Duplicate rows,0,0
5,Columns,12,12


In [11]:
df.to_csv('cleaned_titanic.csv', index=False)
print("Cleaned dataset saved as cleaned_titanic.csv")

Cleaned dataset saved as cleaned_titanic.csv


## Summary

This dataset was cleaned by:
1. Filling 177 missing Age values with the median (28.0)
2. Filling 2 missing Embarked values with the mode ('S')
3. Converting the 77%-missing Cabin column into a binary 'Has_Cabin' feature instead of imputing or dropping it entirely
4. Confirming 0 duplicate rows existed
5. Verifying consistent formatting across all categorical columns
6. Identifying 116 Fare outliers via IQR and retaining them as legitimate high-value tickets
7. Confirming all data types were correctly assigned

The dataset went from 891 rows with 3 columns containing missing data to a fully complete dataset with no missing values, ready for downstream analysis or modeling.